# Homework 3 --- Problem 2

## Part c

In [1]:
import pandas as pd

df = pd.read_csv("winequality-white.csv", sep=";")
df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error
from itertools import product

X = df.drop(columns=["quality"])
y = df["quality"]

# First split into 80% train+val and 20% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
# Then split 80% train+val into 70% train and 10% validation (10/80 = 0.125)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.125, random_state=42
)

val_size = len(X_val)
print(f"Dataset split lengths: Train={len(X_train)}, Val={val_size}, Test={len(X_test)}")

# Define grid search with > validation set size parameter sets
# Need > 490 combinations.

# RF Parameters:
# n_estimators: 5 options
# max_depth: 5 options
# min_samples_split: 5 options
# max_features: 4 options
# Total combinations = 5 * 5 * 5 * 4 = 500 combinations
rf_grid = {
    'n_estimators': [10, 20, 50, 80, 100],
    'max_depth': [3, 5, 8, 12, None],
    'min_samples_split': [2, 5, 10, 15, 20],
    'max_features': [0.4, 0.6, 0.8, 1.0]
}

# KNN Parameters:
# n_neighbors: 35 options (e.g. 1 to 35)
# weights: 2 options
# p: 3 options
# leaf_size: 3 options
# Total combinations = 35 * 2 * 3 * 3 = 630 combinations
knn_grid = {
    'n_neighbors': list(range(1, 36)),
    'weights': ['uniform', 'distance'],
    'p': [1, 2, 3],
    'leaf_size': [10, 20, 30]
}

def custom_grid_search(model_class, grid, X_tr, y_tr, X_v, y_v):
    keys, values = zip(*grid.items())
    experiments = [dict(zip(keys, v)) for v in product(*values)]
    print(f"Running grid search for {model_class.__name__} with {len(experiments)} combinations...")

    best_mse = float('inf')
    best_params = None
    best_model = None

    for params in experiments:
        model = model_class(**params, random_state=42) if 'random_state' in model_class().get_params() else model_class(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_v)
        mse = mean_squared_error(y_v, preds)
        if mse < best_mse:
            best_mse = mse
            best_params = params
            best_model = model

    return best_model, best_params, best_mse

# Run Search
best_rf, best_rf_params, rf_val_mse = custom_grid_search(RandomForestRegressor, rf_grid, X_train, y_train, X_val, y_val)
best_knn, best_knn_params, knn_val_mse = custom_grid_search(KNeighborsRegressor, knn_grid, X_train, y_train, X_val, y_val)

# Evaluate on Test Set
rf_test_mse = mean_squared_error(y_test, best_rf.predict(X_test))
knn_test_mse = mean_squared_error(y_test, best_knn.predict(X_test))

print("\n--- RESULTS ---")
print(f"Best RF Params: {best_rf_params}")
print(f"RF Validation MSE: {rf_val_mse:.5f}")
print(f"RF Test MSE: {rf_test_mse:.5f}\n")

print(f"Best KNN Params: {best_knn_params}")
print(f"KNN Validation MSE: {knn_val_mse:.5f}")
print(f"KNN Test MSE: {knn_test_mse:.5f}\n")

Dataset split lengths: Train=3428, Val=490, Test=980
Running grid search for RandomForestRegressor with 500 combinations...
Running grid search for KNeighborsRegressor with 630 combinations...

--- RESULTS ---
Best RF Params: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'max_features': 0.6}
RF Validation MSE: 0.30984
RF Test MSE: 0.35033

Best KNN Params: {'n_neighbors': 24, 'weights': 'distance', 'p': 1, 'leaf_size': 10}
KNN Validation MSE: 0.38734
KNN Test MSE: 0.45257



#### Answer:

The empirical risk on the validation set is typically lower than on the test set because hyperparameters were selected specifically to optimize validation performance, introducing an optimistic selection bias. The empirical risk on the test set provides an unbiased estimate of the model's true generalization risk because the test data was completely untouched during both model training and hyperparameter tuning.